# Treinamento do MLP

Vamos utilizar a implementação do [Scikit Learn](https://scikit-learn.org/stable/index.html) do [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

In [1]:
!pip install -r requirements.txt

You should consider upgrading via the '/home/cidigital/Documents/PBL/CI-Digital-PBL---Circuitos-Digitais-IV/Scripts/.env/bin/python3 -m pip install --upgrade pip' command.


In [2]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import scipy.io as sio
from nptdms import TdmsFile
import re

In [3]:
acoustic_path = "dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/acoustic"
vibration_path = "dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/vibration"
current_temp_path = "dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/current,temp"

acoustic_files = [f for f in Path(acoustic_path).iterdir() if f.is_file()]
vibration_files = [f for f in Path(vibration_path).iterdir() if f.is_file()]
current_temp_files = [f for f in Path(current_temp_path).iterdir() if f.is_file()]

In [14]:
for file in vibration_files:
    print(file)

dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/vibration/0Nm_BPFI_03.mat
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/vibration/0Nm_BPFO_03.mat
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/vibration/0Nm_Misalign_01.mat
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/vibration/0Nm_Normal.mat
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/vibration/0Nm_Unbalance_0583mg.mat
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/vibration/0Nm_Unbalance_1169mg

In [15]:
for file in current_temp_files:
    print(file)

dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/current,temp/0Nm_BPFI_03.tdms
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/current,temp/0Nm_BPFO_03.tdms
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/current,temp/0Nm_Misalign_01.tdms
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/current,temp/0Nm_Normal.tdms
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/current,temp/0Nm_Unbalance_0583mg.tdms
dataset/Vibration, Acoustic, Temperature, and Motor Current Dataset of Rotating Machine Under Varying Load Conditions for Fault Diagnosis/current,te

In [4]:
dfs = []
for file in current_temp_files:
    data = TdmsFile.read(file)
    log_group = data["Log"]
    df = log_group.as_dataframe()
    first_channel = log_group.channels()[0]
    df["Timestamp"] = first_channel.time_track(absolute_time=True)
    df["file_name"] = file.name
    if "BPFI" in file.name or "BPFO" in file.name:
        error_type = "Bearing"
    elif "Unbalance" in file.name:
        error_type = "Unbalance"
    elif "Misalign" in file.name:
        error_type = "Misalign"
    elif "Normal" in file.name:
        error_type = "None"
    df["Error_Type"] = error_type
    dfs.append(df)

current_temp_combined_df = pd.concat(dfs, ignore_index=True)
current_temp_combined_df.rename(columns={
        "cDAQ9185-1F486B5Mod1/ai0": "Temperature_housing_A",
        "cDAQ9185-1F486B5Mod1/ai1": "Temperature_housing_B",
        "cDAQ9185-1F486B5Mod2/ai0": "U-phase",
        "cDAQ9185-1F486B5Mod2/ai2": "V-phase",
        "cDAQ9185-1F486B5Mod2/ai3": "W-phase"
    }, inplace=True)

In [5]:
current_temp_combined_df.head()

,Temperature_housing_A,Temperature_housing_B,U-phase,V-phase,W-phase,Timestamp,file_name,Error_Type
0,27.607992,28.217591,1.894377,0.949463,-2.165271,2021-06-28 11:02:43.489671000,0Nm_BPFI_03.tdms,Bearing
1,27.607992,28.217591,2.128889,0.926111,-2.309938,2021-06-28 11:02:43.489710050,0Nm_BPFI_03.tdms,Bearing
2,27.607992,28.217591,2.373001,0.882154,-2.526938,2021-06-28 11:02:43.489749100,0Nm_BPFI_03.tdms,Bearing
3,27.607992,28.217591,2.087747,1.193974,-2.444867,2021-06-28 11:02:43.489788150,0Nm_BPFI_03.tdms,Bearing
4,27.607992,28.217591,2.393572,1.125291,-2.718899,2021-06-28 11:02:43.489827200,0Nm_BPFI_03.tdms,Bearing


In [6]:
current_temp_combined_df.size

430217648

In [7]:
import os
import numpy as np
import pandas as pd
import scipy.io as sio

dfs = []
FS = 25600.0  # Sampling frequency fallback (25.6 kHz)

# Ensure Timestamp is datetime in the target reference dataframe
current_temp_combined_df["Timestamp"] = pd.to_datetime(
    current_temp_combined_df["Timestamp"]
)

for file in vibration_files:
    # 1. Handle Path object properly
    file_path = str(file)
    base_name = file.name if hasattr(file, "name") else os.path.basename(file_path)
    tdms_name = base_name.replace(".mat", ".tdms")

    # 2. Load mat file
    data = sio.loadmat(file_path)
    signal_struct = data["Signal"][0, 0]

    # Extract raw signal data
    raw_data = signal_struct["y_values"][0, 0][0]

    if raw_data.ndim == 2 and raw_data.shape[1] == 5:
        rel_timestamps = raw_data[:, 0]
        vibration_data = raw_data[:, 1:]
    else:
        vibration_data = raw_data
        n_samples = len(vibration_data)

        t_start = 0.0
        dt = 1.0 / FS

        if (
            "x_values" in signal_struct.dtype.names
            and signal_struct["x_values"].size > 0
        ):
            x_struct = signal_struct["x_values"][0, 0]
            x_names = x_struct.dtype.names if x_struct.dtype.names else ()

            if "start_value" in x_names:
                t_start = float(np.ravel(x_struct["start_value"])[0])
            if "increment" in x_names:
                dt = float(np.ravel(x_struct["increment"])[0])

        rel_timestamps = t_start + (np.arange(n_samples) * dt)

    channel_names = [
        "x_direction_housing_A",
        "y_direction_housing_A",
        "x_direction_housing_B",
        "y_direction_housing_B",
    ]

    # 3. Build DataFrame
    df = pd.DataFrame(vibration_data, columns=channel_names)
    df["rel_timestamp"] = rel_timestamps
    df["file_name"] = tdms_name

    # 4. Lookup absolute t0 start time
    file_tdms_data = current_temp_combined_df[
        current_temp_combined_df["file_name"] == tdms_name
    ]

    if not file_tdms_data.empty:
        t0_absolute = file_tdms_data["Timestamp"].min()
        df["Timestamp"] = t0_absolute + pd.to_timedelta(
            rel_timestamps, unit="s"
        )
    else:
        df["Timestamp"] = pd.NaT

    dfs.append(df)

# Combine into single DataFrame
df_vibration = pd.concat(dfs, ignore_index=True)

# Important: Drop missing timestamps to satisfy merge_asof requirements
df_vibration = df_vibration.dropna(subset=["Timestamp"]).reset_index(drop=True)

In [8]:
df_vibration.head()

,x_direction_housing_A,y_direction_housing_A,x_direction_housing_B,y_direction_housing_B,rel_timestamp,file_name,Timestamp
0,-8.947968,14.536224,-1.054167,0.969664,0.000012,0Nm_BPFI_03.tdms,2021-06-28 11:02:43.489682659
1,2.321366,-2.065806,-1.840423,3.046795,0.000051,0Nm_BPFI_03.tdms,2021-06-28 11:02:43.489721721
2,0.273554,2.071666,-0.860117,2.146393,0.000090,0Nm_BPFI_03.tdms,2021-06-28 11:02:43.489760784
3,-7.890409,15.446308,-0.465193,1.381279,0.000129,0Nm_BPFI_03.tdms,2021-06-28 11:02:43.489799846
4,-1.295177,-0.780089,0.003352,-0.305207,0.000168,0Nm_BPFI_03.tdms,2021-06-28 11:02:43.489838909


In [9]:
df_vibration.size

333312000

In [10]:
final_df = pd.merge_asof(
    df_vibration.sort_values("Timestamp"),
    current_temp_combined_df.sort_values("Timestamp"),
    on="Timestamp",
    by="file_name",
    direction="nearest",
    tolerance=pd.Timedelta("1s")
)

In [11]:
final_df.head()

,x_direction_housing_A,y_direction_housing_A,x_direction_housing_B,y_direction_housing_B,rel_timestamp,file_name,Timestamp,Temperature_housing_A,Temperature_housing_B,U-phase,V-phase,W-phase,Error_Type
0,1.435439,-0.285463,-1.046865,1.194944,0.000026,0Nm_Normal.tdms,2021-06-25 06:59:53.566427206,24.953918,25.18773,2.117918,0.784624,-2.251514,None
1,0.882802,-0.629166,-0.650744,0.686506,0.000065,0Nm_Normal.tdms,2021-06-25 06:59:53.566466268,24.953918,25.18773,2.061690,0.937100,-2.321066,None
2,0.309371,-0.883415,-0.713592,-0.272134,0.000104,0Nm_Normal.tdms,2021-06-25 06:59:53.566505331,24.953918,25.18773,2.115175,1.029135,-2.428175,None
3,-0.186416,-0.179984,-1.547133,1.124484,0.000143,0Nm_Normal.tdms,2021-06-25 06:59:53.566544393,24.953918,25.18773,2.405915,1.005783,-2.657694,None
4,-1.160443,0.243725,0.381755,-1.104712,0.000182,0Nm_Normal.tdms,2021-06-25 06:59:53.566583456,24.953918,25.18773,2.352430,0.775009,-2.422611,None


In [12]:
final_df.size

619008000